In [1]:
import sys
import os

# Esto le dice al Notebook que busque los scripts en la carpeta src
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from extraccion import Extraccion
from transformacion import Transformacion
from carga import Carga
import pandas as pd

# EXTRACCIÓN: Conexión al origen NoSQL y descarga de colecciones crudas
ext = Extraccion("mongodb://localhost:27017/", "AirbnbBuenosAires")
datos = ext.ejecutar_extraccion_completa()

# TRANSFORMACIÓN: Aplicación de limpieza, normalización de fechas y lógica de negocio
trans = Transformacion(datos)
datos_listos = trans.ejecutar_transformacion_completa()

# COMPATIBILIDAD SQL: Conversión de estructuras complejas a texto plano
for nombre_df in datos_listos:
    df = datos_listos[nombre_df]
    for col in df.columns:
        # Si la columna contiene listas, las transformamos en texto
        if df[col].apply(lambda x: isinstance(x, list)).any():
            datos_listos[nombre_df][col] = datos_listos[nombre_df][col].astype(str)

# CARGA Y PERSISTENCIA: Almacenamiento y auditoría final
loader = Carga()

# Guardamos en la base de datos relacional para consultas futuras
loader.insertar_sqlite(datos_listos)

# Generamos reportes de oficina (Excel) con control de volumen (Sampling)
loader.exportar_excel(datos_listos)

# Verificación de integridad: Comparamos que lo extraído coincida con lo cargado
loader.verificar_carga(datos_listos)

print("\nProceso finalizado exitosamente.")

21:45:41 - [Extraccion] - INFO - Conectado a MongoDB: AirbnbBuenosAires
21:46:24 - [Extraccion] - INFO - Extraccion OK: calendar (9982072 registros)
21:46:25 - [Extraccion] - INFO - Extraccion OK: listings (27348 registros)
21:46:31 - [Extraccion] - INFO - Extraccion OK: reviews (1042702 registros)
21:46:31 - [Transformacion] - INFO - Fase de Transformacion iniciada.
21:46:40 - [Transformacion] - INFO - Fechas normalizadas en calendar
21:46:40 - [Transformacion] - WARNING - ADVERTENCIA: 'price' no existe en calendar
21:46:41 - [Transformacion] - WARNING - ADVERTENCIA: 'price' no existe en listings
21:46:45 - [Transformacion] - INFO - Fechas normalizadas en reviews
21:47:41 - [Carga] - INFO - Carga SQL OK: calendar
21:47:42 - [Carga] - INFO - Carga SQL OK: listings
21:47:48 - [Carga] - INFO - Carga SQL OK: reviews
21:47:56 - [Carga] - WARNING - TABLA GRANDE: calendar excedio limite Excel. Usando muestra.
21:48:38 - [Carga] - INFO - Excel guardado: calendar
21:49:03 - [Carga] - INFO - Ex